# Bereinigung Korpus 1 — Schweizer Legal-Tech-Marketingtexte

Setzt `Eymann_Workflow_Korpus1.md` Schritt 3 um. Input: `Eymann_Korpus1_roh.csv`
(29 Anbieter, Schritt 1+2 abgeschlossen). Ziel-Output: `Eymann_Korpus1_clean.csv`
als Basis für die Zero-Shot-Klassifikation (AP8/Beers 6 Dimensionen).

**Offene Entscheidungen (bereits getroffen, siehe Workflow-Dokument):**
Satz- statt Absatz-Segmentierung (uneinheitliche Absatzstruktur im Rohkorpus),
englischsprachige Seiten werden mitklassifiziert und per Spalte `sprache`
unterscheidbar gehalten.

Ein erster Durchlauf dieser Pipeline wurde bereits mit vereinfachten Mitteln
(Stopwort-Heuristik statt `langdetect`, `difflib` statt `sklearn`-Cosine-
Similarity) ausgeführt, weil diese Bibliotheken in der Cowork-Sandbox nicht
verfügbar waren. Dieses Notebook nutzt die vollständigen Bibliotheken aus
`requirements.txt` und sollte lokal ausgeführt werden, um `Eymann_Korpus1_clean.csv`
verbindlich zu erzeugen bzw. zu verifizieren.

In [ ]:
# Imports
import re
import hashlib
from collections import Counter

import pandas as pd
from langdetect import detect, DetectorFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DetectorFactory.seed = 0  # langdetect ist sonst nicht deterministisch

PFAD_ROH = "../daten/korpus1/Eymann_Korpus1_roh.csv"
PFAD_CLEAN = "../daten/korpus1/Eymann_Korpus1_clean.csv"


## Schritt 1: Boilerplate-Bereinigung

Entfernt bekannte Reste, die trotz `trafilatura`/BeautifulSoup-Extraktion im
`rohtext` verblieben sind — v.a. bei den per Browser-Snapshot erhobenen Seiten,
wo der einfache BS4-Fallback keine Cookie-Consent-Layer oder Skip-Links kennt
(z.B. Libra: ein ganzer OneTrust-Cookie-Dialog; balo.ai: Wix-Gerüsttext
"top of page"/"bottom of page"; DeepLegal: ein Formular-Bestätigungstext am
Dokumentende).

In [ ]:
# Ab diesen Marken folgt nur noch Formular-/Cookie-Consent-Boilerplate, nicht
# mehr Marketingtext -> Rest des Dokuments ab hier abschneiden.
ENDE_MARKER = [
    r"Vielen Dank für Ihre (Anfrage|Nachricht)\.",
    r"View Vendor Details",
    r"Wir verwenden Cookies auf unserer Website",
]

# Einzelne Boilerplate-Phrasen (Skip-Links, Sprachumschalter, Seitenbau-Reste),
# die irgendwo im Text auftauchen können, aber nicht Teil des Marketingtexts sind.
LITERAL_JUNK = [
    "Zum Inhalt springen",
    "Weiter zum Hauptinhalt",
    "Skip to content",
    "Skip to main content",
    "top of page",
    "bottom of page",
    "DE FR IT EN",
    "[`dialog closed`]",
]

# Anbieter-spezifische Fälle: keine generischen Muster, sondern einmalig von
# Hand identifizierte, nicht zur Marketingprosa gehörende Blöcke.
# - Weblaw/LegalTechHub: Startseite ist grösstenteils eine Filter-/Verzeichnis-UI;
#   nach "Become a partner" folgt nur noch eine lange Taxonomie-Liste
#   (Rechtsgebiete, Kategorien, Länder, Zielgruppen mit Zähler-Badges), keine Prosa.
# - DeepLegal: enthält eine komplette Preistabelle/Feature-Matrix (Speicherplan-
#   Stufen, "Nur mit Advanced Box" x10, doppelter Sprach-/Währungsselektor) vor
#   der eigentlichen Tagline am Dokumentende.
SPEZIALFAELLE_ENTFERNEN = {
    "Weblaw/LegalTechHub": [r"Become a partner Areas of law.*"],
    "DeepLegal": [r"Zwei-Faktor-Authentifizierung.*Anwenden Back "],
}

def spezialfall_bereinigen(anbieter, text):
    for muster in SPEZIALFAELLE_ENTFERNEN.get(anbieter, []):
        text = re.sub(muster, "", text, flags=re.IGNORECASE | re.DOTALL)
    return text


def boilerplate_bereinigen(text):
    for marker in ENDE_MARKER:
        m = re.search(marker, text, flags=re.IGNORECASE)
        if m:
            text = text[: m.start()]
    for phrase in LITERAL_JUNK:
        text = re.sub(re.escape(phrase), " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()


df = pd.read_csv(PFAD_ROH)
df["rohtext"] = df.apply(lambda row: spezialfall_bereinigen(row["anbieter"], row["rohtext"] or ""), axis=1)
df["text_bereinigt"] = df["rohtext"].fillna("").apply(boilerplate_bereinigen)
print("Boilerplate-Bereinigung angewendet auf", len(df), "Zeilen")


## Schritt 2: Deduplizierung

Exakte Duplikate über einen Hash des normalisierten Texts; nahe Duplikate über
TF-IDF + Cosine-Similarity (Schwellenwert 0.85 als Ausgangspunkt — bei echten
Treffern anpassen/manuell prüfen, bevor automatisch zusammengeführt wird).

In [ ]:
def normalisieren(t):
    return re.sub(r"\s+", " ", t.lower()).strip()

hashes = df["text_bereinigt"].apply(lambda t: hashlib.sha256(normalisieren(t).encode("utf-8")).hexdigest())
exakte_duplikate = df[hashes.duplicated(keep=False)][["anbieter"]]
print("Exakte Duplikate:", exakte_duplikate["anbieter"].tolist() if not exakte_duplikate.empty else "keine")


In [ ]:
vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform(df["text_bereinigt"])
sim_matrix = cosine_similarity(tfidf)

SCHWELLENWERT_NAHE_DUPLIKATE = 0.85
nahe_duplikate = []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        if sim_matrix[i, j] > SCHWELLENWERT_NAHE_DUPLIKATE:
            nahe_duplikate.append((df.iloc[i]["anbieter"], df.iloc[j]["anbieter"], round(sim_matrix[i, j], 3)))

print("Nahe Duplikate (Cosine-Similarity > 0.85):", nahe_duplikate if nahe_duplikate else "keine")
print("(Hinweis: CASUS/Leya/Lawise.ai wurden bewusst NICHT zusammengeführt trotz")
print(" Rebranding-Redirect — es sind Marketingtexte je eigenständiger Marke/Domain.)")


## Schritt 3: Segmentierung (Satzebene)

Satz- statt Absatz-Segmentierung (siehe Begründung oben / Workflow-Dokument).
Sehr kurze Fragmente (< 15 Zeichen, meist Navigationsreste) werden verworfen.

In [ ]:
SATZENDE = re.compile(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ0-9])")

def in_saetze_segmentieren(text):
    roh_saetze = SATZENDE.split(text)
    return [s.strip() for s in roh_saetze if len(s.strip()) >= 15]

df["saetze"] = df["text_bereinigt"].apply(in_saetze_segmentieren)
print("Segmente pro Anbieter (min/max):", df["saetze"].apply(len).min(), df["saetze"].apply(len).max())


### Intra-Dokument-Duplikate entfernen

Bei 8 von 29 Anbietern (u.a. Jurata 25x, CASUS 19x, Lawise.ai 11x) kamen exakt
identische Sätze mehrfach im selben Dokument vor — vermutlich Responsive-
Design-Duplikate (Mobile-/Desktop-Version derselben Inhalte beide im HTML) oder
doppelt im DOM vorhandene Karussell-/Testimonial-Widgets. Das verzerrt
Wortfrequenzen und Segmentanzahl pro Anbieter spürbar und wird daher entfernt
(pro Anbieter nur der erste Treffer je exaktem Satztext, Reihenfolge bleibt
erhalten).

In [ ]:
def ohne_interne_duplikate(saetze):
    gesehen = set()
    ergebnis = []
    for satz in saetze:
        if satz not in gesehen:
            gesehen.add(satz)
            ergebnis.append(satz)
    return ergebnis

vorher = df["saetze"].apply(len).sum()
df["saetze"] = df["saetze"].apply(ohne_interne_duplikate)
nachher = df["saetze"].apply(len).sum()
print(f"{vorher - nachher} Intra-Dokument-Duplikate entfernt ({vorher} -> {nachher} Segmente)")


### Ausschluss: Weblaw/LegalTechHub

Nach Bereinigung der Filter-/Taxonomie-Liste (siehe Spezialfall oben) bleiben
für Weblaw/LegalTechHub nur 2 Segmente / 30 Wörter echter Marketingtext übrig —
zu dünn für eine sinnvolle Klassifikation über die 6 Beer-Dimensionen. Weblaw
ist zudem strukturell ein Sonderfall (Verzeichnis/Marktplatz vieler Anbieter,
nicht Eigenwerbung eines einzelnen Produkts wie bei den anderen 28). Entscheid
(22.07.2026, siehe `Eymann_Anbieterliste_Korpus1.md`): aus Korpus 1
ausschliessen. Korpus 1 umfasst damit final 28 Anbieter.

In [ ]:
AUSGESCHLOSSENE_ANBIETER = ["Weblaw/LegalTechHub"]

vorher_anbieter = df["anbieter"].nunique()
df = df[~df["anbieter"].isin(AUSGESCHLOSSENE_ANBIETER)].reset_index(drop=True)
print(f"{vorher_anbieter} -> {df['anbieter'].nunique()} Anbieter (ausgeschlossen: {AUSGESCHLOSSENE_ANBIETER})")


## Schritt 4: Spracherkennung

Pro Satz-Segment (nicht pro ganzem Dokument), da einzelne Seiten mehrsprachige
Abschnitte enthalten (z.B. Sprachumschalter-Reste, DE/EN gemischte Inhalte).

In [ ]:
def sprache_erkennen(text):
    try:
        return detect(text)
    except Exception:
        return "unbekannt"

clean_rows = []
next_id = 1
for _, row in df.iterrows():
    for satz in row["saetze"]:
        clean_rows.append({
            "id": next_id,
            "anbieter": row["anbieter"],
            "text": satz,
            "quelle_url": row["url"],
            "seitentyp": row["seitentyp"],
            "sprache": sprache_erkennen(satz),
            "segment_typ": "satz",
        })
        next_id += 1

df_clean = pd.DataFrame(clean_rows)
print(f"{len(df)} Anbieter -> {len(df_clean)} Satz-Segmente")
print("Sprachverteilung vor Korrektur:", Counter(df_clean['sprache']))


### Kurztext-Korrektur der Spracherkennung

`langdetect` ist bei sehr kurzen Segmenten (Preiszeilen, 3-5-Wort-Fragmente wie
"No hidden costs." oder "Was ist DeepLegal?") unzuverlässig und tippt dann
gerne auf exotische Sprachen (Walisisch, Schwedisch, Tagalog etc.), die im
Korpus faktisch nicht vorkommen — das Korpus ist realistisch nur DE/EN/(FR).
Alles ausserhalb {de, en, fr} wird daher mit einer einfachen, auf diese drei
Sprachen beschränkten Marker-Heuristik nachklassifiziert (Default EN als
Korpus-Mehrheitssprache, falls keine Marker greifen).

In [ ]:
def de_en_fr_heuristik(text):
    t = f" {text.lower()} "
    de_marker = [" und ", " ist ", " für ", " nicht ", " werden ", " können ", " auch ", " sind ",
                 " wie ", " sie ", " wir ", " ihr ", " kein", " jede", " antwort", " kosten"]
    fr_marker = [" et ", " est ", " pour ", " vous ", " nous ", " votre ", " dans ", " avec ", " ses ", " que "]
    en_marker = [" the ", " and ", " for ", " with ", " your ", " you ", " our ", " from ", " this ",
                 " no ", " by ", " lawyers", " time", " full ", " zero "]
    de_score = sum(t.count(m) for m in de_marker) + sum(text.count(c) for c in "äöüßÄÖÜ")
    fr_score = sum(t.count(m) for m in fr_marker)
    en_score = sum(t.count(m) for m in en_marker)
    scores = {"de": de_score, "fr": fr_score, "en": en_score}
    bestimmt = max(scores, key=scores.get)
    return bestimmt if scores[bestimmt] > 0 else "en"

erlaubt = {"de", "en", "fr"}
maske_korrektur = ~df_clean["sprache"].isin(erlaubt)
anzahl_korrigiert = maske_korrektur.sum()
df_clean.loc[maske_korrektur, "sprache"] = df_clean.loc[maske_korrektur, "text"].apply(de_en_fr_heuristik)

print(f"{anzahl_korrigiert} Segmente korrigiert (Kurztext-Heuristik statt langdetect-Ausreisser)")
print("Sprachverteilung nach Korrektur:", Counter(df_clean["sprache"]))

df_clean.to_csv(PFAD_CLEAN, index=False)


## Schritt 4: Deskriptive Statistik

Segmente pro Anbieter, Wortanzahl-Verteilung, Sprachanteil — als Grafiken nach
`outputs/` sowie eine Kontrolle auf sehr lange, satzzeichenfreie Segmente
(z.B. Preistabellen/Feature-Listen ohne Punkt/Ausrufezeichen, die der
Satz-Splitter dadurch nicht weiter zerlegen konnte).

In [ ]:
import matplotlib.pyplot as plt

PFAD_OUTPUTS = "../outputs"

df_clean["wortanzahl"] = df_clean["text"].fillna("").apply(lambda t: len(t.split()))

# 1. Segmente pro Anbieter
segmente_pro_anbieter = df_clean.groupby("anbieter").size().sort_values()
fig, ax = plt.subplots(figsize=(8, 10))
segmente_pro_anbieter.plot(kind="barh", ax=ax, color="#3b6ea5")
ax.set_xlabel("Anzahl Satz-Segmente")
ax.set_title(f"Korpus 1: Satz-Segmente pro Anbieter (n=29, gesamt {len(df_clean)})")
plt.tight_layout()
plt.savefig(f"{PFAD_OUTPUTS}/Eymann_Korpus1_segmente_pro_anbieter.png", dpi=150)
plt.show()

# 2. Wortanzahl-Verteilung pro Segment
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(df_clean["wortanzahl"], bins=range(0, int(df_clean["wortanzahl"].max()) + 2, 1),
        color="#3b6ea5", edgecolor="white")
ax.set_xlabel("Wörter pro Satz-Segment")
ax.set_ylabel("Anzahl Segmente")
ax.set_title("Korpus 1: Wortanzahl-Verteilung pro Satz-Segment")
plt.tight_layout()
plt.savefig(f"{PFAD_OUTPUTS}/Eymann_Korpus1_wortanzahl_verteilung.png", dpi=150)
plt.show()

# 3. Sprachanteil
sprachanteil = df_clean["sprache"].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(sprachanteil.values, labels=[f"{s} ({v})" for s, v in sprachanteil.items()],
       autopct="%1.1f%%", colors=["#3b6ea5", "#a5c8e8", "#c9484f"])
ax.set_title("Korpus 1: Sprachanteil (Satz-Segmente)")
plt.tight_layout()
plt.savefig(f"{PFAD_OUTPUTS}/Eymann_Korpus1_sprachanteil.png", dpi=150)
plt.show()

print("Segmente pro Anbieter:")
print(segmente_pro_anbieter.to_string())
print()
print("Wortanzahl pro Segment, Kennzahlen:")
print(df_clean["wortanzahl"].describe().round(1).to_string())
print()
print("Sprachanteil:")
print(sprachanteil.to_string())


In [ ]:
# Kontrolle: sehr lange Segmente (>100 Wörter) deuten auf satzzeichenfreie
# Listen/Tabellen hin, die der Satz-Splitter nicht weiter zerlegen konnte.
lange_segmente = df_clean[df_clean["wortanzahl"] > 100][["anbieter", "wortanzahl", "text"]]
print(f"{len(lange_segmente)} von {len(df_clean)} Segmenten >100 Wörter (unsegmentierte Listen/Tabellen):")
print(lange_segmente[["anbieter", "wortanzahl"]].to_string(index=False))


## Nächste Schritte

1. Die wenigen sehr langen Segmente (>100 Wörter, satzzeichenfreie Listen/
   Tabellen wie Preistabellen) sind ein akzeptierter Rest — betrifft <1% des
   Korpus, keine weitere Bereinigung vorgesehen; bei Bedarf in der
   Analysephase gesondert markieren.
2. Falls nahe Duplikate gefunden wurden: manuell entscheiden (zusammenführen,
   nur eine Version behalten, oder als eigenständig belassen — z.B. bei
   Anbietern mit mehreren Marken/Produktlinien).
3. Weiter mit AP7 (Methodenentscheid) aus `Workflow_Arbeitspakete.md`.